<a href="https://colab.research.google.com/github/diegocortes30/pharma_batch_quality_analysis/blob/main/02_sql_layer_pharma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pharma Batch Quality Analysis: 02 SQL Layer**

**Goal:** Explore the dataset with SQL: counts, aggregations and joins between the three tables as a first approach to the business questions.

**Data:** [Big data collection in pharmaceutical manufacturing](https://figshare.com/collections/Cholessterol-Lowering_Drug_Process_and_Quality_Data/5645578) (figshare), 3 CSV files: Laboratory, Process, Normalization.

**Source:** Žagar, J. & Mihelič, J. (2022). Big data collection in pharmaceutical manufacturing and its use for product quality predictions. Scientific Data, 9(1), 99. https://doi.org/10.1038/s41597-022-01203-x

**Contents:** setup and database creation, basic queries, aggregations, joins, export queries, findings.

## 1. Setup and database creation

In [23]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sqlite3
import pandas as pd

RAW = '/content/drive/MyDrive/pharma_batch_quality_analysis/data/raw'
DB_PATH = '/content/drive/MyDrive/pharma_batch_quality_analysis/pharma.db'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
db_exists = os.path.exists(DB_PATH)
conn = sqlite3.connect(DB_PATH)

if not db_exists:
    for file in os.listdir(RAW):
        if file.endswith('.csv'):
            name = file.replace('.csv', '')
            df = pd.read_csv(f'{RAW}/{file}', sep=';')
            df.to_sql(name, conn, if_exists='replace', index=False)
            print(f'{name}: {len(df):,} rows loaded')
else:
    print('Database already exists, skipping CSV load.')

Database already exists, skipping CSV load.


In [25]:
def query(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn)

# Check that the 3 tables exist
query("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,Laboratory
1,Process
2,Normalization


## 2. Basic queries

In [26]:
# How many batches exist per dose
base_batches_per_dose = """
SELECT strength, COUNT(*) AS batches
FROM Laboratory
GROUP BY strength
ORDER BY batches DESC
"""
query(base_batches_per_dose)

,strength,batches
0,20M,445
1,5MG,249
2,10M,213
3,40M,98


In [27]:
# How many batches exist per api code
base_batches_per_api = """
SELECT api_code, COUNT(*) AS batches
FROM Laboratory
GROUP BY api_code
"""
query(base_batches_per_api)

,api_code,batches
0,1,150
1,2,15
2,3,481
3,4,24
4,5,335


In [28]:
# How many batches had pause at the weekend
base_batches_weekend = """
SELECT weekend, COUNT(*) AS batches
FROM Process
GROUP BY weekend
"""
query(base_batches_weekend)

,weekend,batches
0,no,950
1,yes,55


## 3. Aggregations

In [29]:
# Quality summary per dose
q1_quality_by_dose = """
SELECT strength,
       COUNT(*) AS batches,
       ROUND(AVG(dissolution_min), 2) AS avg_dissolution_min,
       MIN(dissolution_min) AS min_dissolution_min,
       MAX(dissolution_min) AS max_dissolution_min,
       ROUND(AVG(impurities_total), 2) AS avg_impurities_total,
       MIN(impurities_total) AS min_impurities_total,
       MAX(impurities_total) AS max_impurities_total
FROM Laboratory
GROUP BY strength
"""
query(q1_quality_by_dose)

,strength,batches,avg_dissolution_min,min_dissolution_min,max_dissolution_min,avg_impurities_total,min_impurities_total,max_impurities_total
0,10M,213,85.75,77,95,0.13,0.05,0.60
1,20M,445,84.11,74,98,0.17,0.05,0.40
2,40M,98,82.78,75,97,0.08,0.05,0.34
3,5MG,249,89.20,81,100,0.13,0.05,0.37


In [30]:
# Quality summary per api_code
q2_quality_by_api = """
SELECT api_code,
       COUNT(*) AS batches,
       ROUND(AVG(dissolution_min), 2) AS avg_dissolution_min,
       MIN(dissolution_min) AS min_dissolution_min,
       MAX(dissolution_min) AS max_dissolution_min,
       ROUND(AVG(impurities_total), 2) AS avg_impurities_total,
       MIN(impurities_total) AS min_impurities_total,
       MAX(impurities_total) AS max_impurities_total
FROM Laboratory
GROUP BY api_code
"""
query(q2_quality_by_api)

,api_code,batches,avg_dissolution_min,min_dissolution_min,max_dissolution_min,avg_impurities_total,min_impurities_total,max_impurities_total
0,1,150,87.58,77,100,0.07,0.05,0.23
1,2,15,85.20,80,89,0.11,0.06,0.24
2,3,481,85.39,74,98,0.08,0.05,0.60
3,4,24,87.83,81,97,0.11,0.05,0.26
4,5,335,84.84,74,98,0.26,0.17,0.40


In [31]:
# Count batches with low dissolution (< 80%) per dose
q1_low_dissolution_by_dose = """
SELECT strength, COUNT(*) AS batches
FROM Laboratory
WHERE dissolution_min < 80
GROUP BY strength
ORDER BY batches DESC
"""
query(q1_low_dissolution_by_dose)

,strength,batches
0,20M,33
1,40M,14
2,10M,3


## 4. Joins

In [32]:
# Compression force and dissolution per dose (Laboratory + Process)
q3_compression_by_dose = """
SELECT l.strength,
       COUNT(*) AS batches,
       ROUND(AVG(p."main_CompForce mean"), 2) AS avg_comp_force,
       ROUND(AVG(l.dissolution_min), 2) AS avg_dissolution_min
FROM Laboratory AS l
JOIN Process AS p
    ON l.batch = p.batch
GROUP BY l.strength
"""
query(q3_compression_by_dose)

,strength,batches,avg_comp_force,avg_dissolution_min
0,10M,213,5.30,85.75
1,20M,445,6.45,84.11
2,40M,98,9.60,82.78
3,5MG,249,4.56,89.20


In [33]:
# Quality with and without a weekend pause (Laboratory + Process)
q3_quality_by_weekend = """
SELECT p.weekend,
       COUNT(*) AS batches,
       ROUND(AVG(l.impurities_total), 2) AS avg_total_impurities,
       ROUND(AVG(l.dissolution_min), 2) AS avg_dissolution_min
FROM Laboratory AS l
JOIN Process AS p
    ON l.batch = p.batch
GROUP BY p.weekend
"""
query(q3_quality_by_weekend)

,weekend,batches,avg_total_impurities,avg_dissolution_min
0,no,950,0.14,85.61
1,yes,55,0.13,85.20


In [34]:
# Average normalized start-up waste per product (Process + Normalization)
q3_startup_waste_by_product = """
SELECT p.code,
       COUNT(*) AS batches,
       ROUND(AVG(p.startup_waste / n."Normalisation factor"), 2) AS avg_normalized_waste
FROM Process AS p
JOIN Normalization AS n
    ON p.code = n."Product code"
GROUP BY p.code
ORDER BY avg_normalized_waste DESC
"""
query(q3_startup_waste_by_product)

,code,batches,avg_normalized_waste
0,16,2,1739.38
1,9,4,1655.00
2,4,22,1630.94
3,14,11,1491.57
4,8,2,1249.32
5,21,68,1219.84
6,1,95,1107.55
7,25,34,888.54
8,23,187,844.05
9,11,15,829.75


## 5. Export queries to .sql files

Each analysis query is saved as a `.sql` file so it can be reused outside the notebook and shown in the `sql/` folder of the repository.

In [35]:
sql_dir = '/content/drive/MyDrive/pharma_batch_quality_analysis/sql'
os.makedirs(sql_dir, exist_ok=True)

queries = {
    'base_batches_per_dose': base_batches_per_dose,
    'base_batches_per_api': base_batches_per_api,
    'base_batches_weekend': base_batches_weekend,
    'q1_quality_by_dose': q1_quality_by_dose,
    'q1_low_dissolution_by_dose': q1_low_dissolution_by_dose,
    'q2_quality_by_api': q2_quality_by_api,
    'q3_compression_by_dose': q3_compression_by_dose,
    'q3_quality_by_weekend': q3_quality_by_weekend,
    'q3_startup_waste_by_product': q3_startup_waste_by_product,
}

for name, sql in queries.items():
    with open(f'{sql_dir}/{name}.sql', 'w') as f:
        f.write(sql.strip() + ';\n')
    print(f'{name}.sql saved')

base_batches_per_dose.sql saved
base_batches_per_api.sql saved
base_batches_weekend.sql saved
q1_quality_by_dose.sql saved
q1_low_dissolution_by_dose.sql saved
q2_quality_by_api.sql saved
q3_compression_by_dose.sql saved
q3_quality_by_weekend.sql saved
q3_startup_waste_by_product.sql saved


## 6. Findings
**1. Batches per strength:** 20 mg has the most batches (445), followed by 5 mg (249) and 10 mg (213), while 40 mg has only 98.

**2. Batches per API source:** API source 3 has the most batches (481), followed by 5 (335) and 1 (150). Sources 4 (24) and 2 (15) have the fewest.

**3. Dissolution by dose (Q1):** minimum dissolution (% of API released in 30 minutes) decreases as the dose increases: 5 mg (89.20%), 10 mg (85.75%), 20 mg (84.11%) and 40 mg (82.78%).
50 batches (~5%) have a minimum dissolution below 80% (a reference threshold, not the specification): 33 of 20 mg, 14 of 40 mg (the highest proportion, 14% of its batches), 3 of 10 mg and none of 5 mg.

**4. Compression force by dose (Q3):** compression force increases with the dose: 5 mg (4.56 kN), 10 mg (5.30 kN), 20 mg (6.45 kN) and 40 mg (9.60 kN). Since force and dissolution both change with the dose, it is not yet possible to say whether force affects dissolution; this will be analyzed within each dose in notebook 04.

**5. Impurities by API source (Q2):** API source 5 has much higher total impurities (average 0.26%, minimum 0.17%) than the other sources (averages of 0.07–0.11%). This is an observation, not a cause: source 5 might be related to specific doses or periods.

**6. Weekend pause:** no meaningful difference was observed in minimum dissolution (85.61% vs 85.20%) or total impurities (0.14% vs 0.13%) between batches with and without a weekend pause.

**7. Normalized start-up waste:** start-up waste varies widely between products. The two products with the highest values (codes 16 and 9) have only 2 and 4 batches, which is too few to draw conclusions. Among products with 30+ batches, waste ranges from ~320 (codes 13 and 22) to ~1,220 (code 21). Results for codes 14 and 16 depend on the batch size decision from notebook 01: if their real size were 4,800,000, their normalized waste would be 10 times lower.